Get OpenAI client

In [4]:
from openai import OpenAI
openai_client = OpenAI()

Ask standard GPT LLM a question 

In [3]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

print(response.output_text)

Yes, likely — but it depends on the course’s enrollment rules and whether it’s still open.

If you want, send me:
- the course name
- the platform or school
- whether you want to join as a student, auditor, or late enrollee

and I can help you figure out the exact next step or draft a message to the instructor/admin.


Download the FAQ data through load_faq_data function

In [6]:
from ingest import load_faq_data, build_index
documents = load_faq_data()
index = build_index(documents)

Define a search function which encapsultes the

In [3]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search('how do i run ollama ?')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

Tell the GPT model about the actual search function

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

Tell the LLM about the search function so that it considers it as a tool it can use

In [11]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)

In [15]:
call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join new student enrollment registration"}', call_id='call_yHTzBfWGAcDAhsLtPDxoAWfE', name='search', type='function_call', id='fc_0b0c0be1a9384441006ad8b62a879887d292ccf88b3081bc20', async_=None, caller=None, namespace=None, status='completed')

In [15]:
import json

args = json.loads(call.arguments)
args

NameError: name 'call' is not defined

In [24]:
results = search(**args)
results_json = json.dumps(results, indent=2)
print(results_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "1e829a8c6f",
    "course": "llm-zoomcamp",
    "section": "Module 2: Vector Search",
    "question": "Do I need a new GitHub repo for Module 2, or just a new codespace?",
    "answer": "Just a new codespace. A codespace is an environment (see *Can I run the course locally instead of Codespaces?*); you create it from your existing repository, so you don't need a new GitHub repo.\n\nUse a separate codespace for Module 2 because the vector-search dependencies are fairly heavy. Keeping them isolated means you can simply stop or delete that codespace when you're done, rather than leaving the extra weight in your Module 1 environment. Setup is quick.\n\nSince

These results are sent back to the LLM

In [26]:
messages.extend(response.output)

# Append the function call output to the messages list to provide a full history to the LLM
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": results_json,
})

In [34]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join new student enrollment registration"}', call_id='call_yHTzBfWGAcDAhsLtPDxoAWfE', name='search', type='function_call', id='fc_0b0c0be1a9384441006ad8b62a879887d292ccf88b3081bc20', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_yHTzBfWGAcDAhsLtPDxoAWfE',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "1e829a8c6f",\n    "course": "llm-zoomcamp",\n    "section": "Module 2: Vector Search",\n    "question": "Do I need a new GitHub repo for Module 2, or jus

In [36]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Yes, you can still join and start learning.

If you want a certificate, you’ll need to submit your project while the course is still accepting submissions.


## Global agent steps 
1. Make a call to the LLM (let him make the choice to use the search function or not) <-- first call to the LLM
2. LLM decides to invoke search('params')
3. We invoke the search, we have the results 
4. Send the results back to the LLM <-- second call to the LLM
5. LLM procesees the results 
6. LLM gives the answer

### Agentic loop

In [12]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "I just discoverd the course. Can I join it?"

messages = [
    {'role': 'developer', 'content': instructions}
]

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)

In [13]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [22]:
# This section handles the interaction with the assistant, including sending the initial question,
# processing the response, and handling any function calls that the assistant may make.
question = "Is this course useful to become an AI engineer noting that I am a data enginer?"

# Prepare the initial messages for the assistant, including the developer instructions and the user's question.
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

# Send the initial request to the assistant with the prepared messages and available tools.
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

# Extend the message history with the assistant's response and initialize the function call flag.
messages.extend(response.output)
has_function_calls = False

# Process each item in the assistant's response, handling function calls and messages accordingly.
for item in response.output:
    # Check if the current item is a function call and handle it accordingly.
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    # Handle the case where the assistant's response is a regular message.
    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"AI engineer data engineer course useful FAQ"}
function_call: search {"query":"data engineer become AI engineer course FAQ"}
function_call: search {"query":"career transition data engineer AI engineer machine learning course FAQ"}


In [20]:
print(response)

Response(id='resp_0f8bb9cae94f22c5006ad8da8b277087d2be06191ef6bd9819', access_programs=None, created_at=1792596619.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment new student"}', call_id='call_KtojReHRAr9nagEyD29DF8DM', name='search', type='function_call', id='fc_0f8bb9cae94f22c5006ad8da8cd0a887d287771ade3e9c03f5', async_=None, caller=None, namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"course FAQ enrollment registration late join discovered course"}', call_id='call_sa2yYRG72hncHTRwwpZAZJyu', name='search', type='function_call', id='fc_0f8bb9cae94f22c5006ad8da8cd0b887d295b5a198750027dd', async_=None, caller=None, namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"can I join the course after it started FAQ"}', call_id='call_EQaZGfRhczgFqMP47

In [ ]:
# This section handles the iterative interaction with the assistant, processing function calls until none remain.
it = 1

# Initialize the iteration counter and start the loop for handling function calls.
while True:
    print(f"iteration #{it}...")
    has_function_calls = False 

    # Call the assistant with the current message history and available tools.
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    # Extend messages to append the assistant's response to the conversation history.
    messages.extend(response.output)

    # Process each item in the assistant's response, handling function calls and messages accordingly.
    for item in response.output:
        # Check if the current item is a function call and handle it accordingly.
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        # Handle the case where the assistant's response is a regular message.
        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    # Increment the iteration counter and check if there are any remaining function calls.
    it = it + 1

    # If there are no function calls in the current iteration, exit the loop.
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — if your goal is to become an AI engineer, this course can be very useful, especially coming from a data engineering background.

Why it’s a good fit for you:
- Data engineers already have strong fundamentals in pipelines, storage, ETL/ELT, orchestration, and working with structured systems.
- AI engineering often builds on those skills, but adds:
  - model/API integration
  - retrieval and vector search
  - prompt engineering
  - evaluation
  - deployment/orchestration of AI apps
- This course appears focused on practical LLM/AI engineering topics, which are directly relevant to modern AI engineer roles.

For a data engineer, this is especially valuable because:
- you likely already understand data infrastructure, which is a big advantage
- you may need to fill gaps in ML/LLM workflows, experimentation, and application-level AI
- many AI engineer roles are really “software + data + LLM systems” roles

That said, whether it’s “enough” depends on your tar

In [24]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results, and then perform more searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

In [26]:
# This section handles the iterative interaction with the assistant, processing function calls until none remain.

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

# This section handles the interaction with the assistant, including sending the initial question,
# processing the response, and handling any function calls that the assistant may make.
question = "Is this course useful to become an AI engineer noting that I am a data enginer?"

# Prepare the initial messages for the assistant, including the developer instructions and the user's question.
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

# Initialize the iteration counter and start the loop for handling function calls.
while True:
    print(f"iteration #{it}...")
    has_function_calls = False 

    # Call the assistant with the current message history and available tools.
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    # Extend messages to append the assistant's response to the conversation history.
    messages.extend(response.output)

    # Process each item in the assistant's response, handling function calls and messages accordingly.
    for item in response.output:
        # Check if the current item is a function call and handle it accordingly.
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        # Handle the case where the assistant's response is a regular message.
        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    # Increment the iteration counter and check if there are any remaining function calls.
    it = it + 1

    # If there are no function calls in the current iteration, exit the loop.
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"AI engineer data engineer course useful become AI engineer prerequisite data engineer"}
iteration #2...
function_call: search {"query":"data engineer AI engineer career transition course suited for data engineers machine learning engineer ai engineer"}
iteration #3...
ASSISTANT:
Yes — this course should be useful for you as a data engineer if your goal is to move toward AI engineering, especially in the LLM / GenAI space.

Why it fits:
- It covers practical AI application building, not just theory.
- You can use free or low-cost APIs to complete the work, so it’s approachable while learning.
- The course expects you to understand the solutions you build, which is valuable if you want to work as an AI engineer, not just use tools superficially.
- Since you already have data engineering experience, you’ll likely have an advantage in data pipelines, integration, orchestration, and production mindset — all useful in AI systems work.

A good f

In [27]:
# This section handles the iterative interaction with the assistant, processing function calls until none remain.

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results, and then perform more searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

# This section handles the interaction with the assistant, including sending the initial question,
# processing the response, and handling any function calls that the assistant may make.
question = "Is this course useful to become an AI engineer noting that I am a data enginer?"

# Prepare the initial messages for the assistant, including the developer instructions and the user's question.
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

# Initialize the iteration counter and start the loop for handling function calls.
while True:
    print(f"iteration #{it}...")
    has_function_calls = False 

    # Call the assistant with the current message history and available tools.
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    # Extend messages to append the assistant's response to the conversation history.
    messages.extend(response.output)

    # Process each item in the assistant's response, handling function calls and messages accordingly.
    for item in response.output:
        # Check if the current item is a function call and handle it accordingly.
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        # Handle the case where the assistant's response is a regular message.
        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    # Increment the iteration counter and check if there are any remaining function calls.
    it = it + 1

    # If there are no function calls in the current iteration, exit the loop.
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"course useful become AI engineer data engineer AI engineer data engineering FAQ"}
iteration #2...
function_call: search {"query":"AI engineer data engineer useful transition from data engineer to AI engineer course project RAG orchestration deployment FAQ"}
iteration #3...
ASSISTANT:
Yes — this course can be useful for becoming an AI engineer, especially if you already work as a data engineer.

Why it fits your background:
- As a data engineer, you likely already know pipelines, data quality, orchestration, and production workflows.
- This course seems to focus on practical LLM/AI engineering topics like:
  - RAG
  - orchestration
  - AI workflows
  - project/capstone work
- Those are exactly the kinds of skills that help bridge from data engineering into AI engineering.

A few important notes:
- It’s most useful if you want to work on applied AI systems, not just model research.
- You should be able to understand and explain what you bui

In [ ]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        # If there are no function calls in this iteration, we can break the loop.
        if has_function_calls == False:
            break

    return last_answer

In [30]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "Is this course enough to build a commercial agent which will try to find prospects to contact for architecture data/AI missions ?"

answer = agent_loop(instructions, question)
print(answer)

iteration #1...
function_call: search {"query":"commercial agent find prospects contact architecture data AI missions course enough"}
function_call: search {"query":"course build agent prospecting contacting clients architecture data AI missions"}
function_call: search {"query":"FAQ commercial agent prospects contact sales outreach AI architecture data"}
iteration #2...
ASSISTANT:
Short answer: **probably enough to build a first version, but not enough by itself to make it production-ready and commercially safe**.

This course should help you build the core pieces of a commercial agent, especially if it covers:
- RAG / search over internal and external data
- tool use / agent orchestration
- evaluation
- basic deployment patterns

But for a real prospecting agent that contacts potential architecture data/AI clients, you’ll likely also need to handle:

- **Lead sourcing and enrichment**: company databases, LinkedIn-like workflows, website scraping, contact discovery
- **Qualification lo

### Use framework

In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [6]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [7]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [8]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [9]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [18]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interfact,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [19]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [20]:
result.cost

CostInfo(input_cost=Decimal('0.000471'), output_cost=Decimal('0.001494'), total_cost=Decimal('0.001965'))

In [21]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Ollama run locally install start server local model FAQ"}', call_id='call_uLZ8BsRx3c1UbjYRQ4LRaU1S', name='search', type='function_call', id='fc_02a5913ea16417ba006ada05e5f82887d28b8e234bc76a951d', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_uLZ8B

In [22]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [23]:
runner.run()

-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None), EasyInputMessage(content='how do I run Docker ?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"Docker run how to run Docker FAQ install start container command"}', call_id='call_YJDrtRXfNwbRi0m99ZBmrMTE', name='search', type='function_call', id='fc_0f0d4b9c6b97e945006ada08b7f4c887d289d62860856112a7', async_=None, caller=None, namespace=None, status='completed'), ResponseFunctionToolCall(argument